In [1]:
import pandas as pd

In [2]:
data = {
    "customer_id": [
        "C101", "C102", "C101", "C103",
        "C102", "C101", "C103", "C102",
        "C101", "C103", "C102", "C103"
    ],
    "month": [
        "2026-01-01", "2026-01-01", "2026-02-01", "2026-01-01",
        "2026-02-01", "2026-03-01", "2026-02-01", "2026-03-01",
        "2026-04-01", "2026-03-01", "2026-04-01", "2026-04-01"
    ],
    "plan": [
        "Basic", "Pro", "Basic", "Basic",
        "Pro", "Pro", "Basic", "Basic",
        "Pro", "Pro", "Basic", "Pro"
    ],
    "mrr": [
        1000, 2500, 1000, 800,
        3000, 1800, 1000, 1500,
        2000, 1600, 1500, 1600
    ]
}

subscriptions = pd.DataFrame(data)

In [3]:
subscriptions.head()

,customer_id,month,plan,mrr
0,C101,2026-01-01,Basic,1000
1,C102,2026-01-01,Pro,2500
2,C101,2026-02-01,Basic,1000
3,C103,2026-01-01,Basic,800
4,C102,2026-02-01,Pro,3000


In [4]:
subscriptions.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  12 non-null     str  
 1   month        12 non-null     str  
 2   plan         12 non-null     str  
 3   mrr          12 non-null     int64
dtypes: int64(1), str(3)
memory usage: 516.0 bytes


In [5]:
subscriptions[subscriptions.duplicated(subset=['customer_id', 'month'])]

,customer_id,month,plan,mrr


In [6]:
subscriptions['month'] = pd.to_datetime(subscriptions['month'])

In [7]:
subscriptions.head()

,customer_id,month,plan,mrr
0,C101,2026-01-01,Basic,1000
1,C102,2026-01-01,Pro,2500
2,C101,2026-02-01,Basic,1000
3,C103,2026-01-01,Basic,800
4,C102,2026-02-01,Pro,3000


Sorting values based on month

In [8]:
subscriptions = subscriptions.sort_values(['customer_id', 'month'])

In [9]:
subscriptions

,customer_id,month,plan,mrr
0,C101,2026-01-01,Basic,1000
2,C101,2026-02-01,Basic,1000
5,C101,2026-03-01,Pro,1800
8,C101,2026-04-01,Pro,2000
1,C102,2026-01-01,Pro,2500
4,C102,2026-02-01,Pro,3000
7,C102,2026-03-01,Basic,1500
10,C102,2026-04-01,Basic,1500
3,C103,2026-01-01,Basic,800
6,C103,2026-02-01,Basic,1000


Build a curated customer-month dataset containing:

1. previous_mrr — customer's MRR from the previous available month.
2. mrr_change — current MRR minus previous MRR.
3. mrr_change_pct — percentage change from previous MRR, expressed as a percentage.
4. lifetime_revenue — cumulative MRR collected from that customer up through the current row.
5. subscription_month_number — 1, 2, 3... independently for each customer.

In [10]:
subscriptions['previous_mrr'] = subscriptions.groupby('customer_id')['mrr'].shift(1)

In [11]:
subscriptions['mrr_change'] = subscriptions.groupby('customer_id')['mrr'].diff()

In [12]:
subscriptions['mrr_change_pct'] = subscriptions.groupby('customer_id')['mrr'].pct_change() * 100

In [13]:
subscriptions['lifetime_revenue'] = subscriptions.groupby('customer_id')['mrr'].cumsum()

In [14]:
subscriptions['subscription_month_number'] = subscriptions.groupby('customer_id')['mrr'].cumcount() + 1

In [15]:
subscriptions

,customer_id,month,plan,mrr,previous_mrr,mrr_change,mrr_change_pct,lifetime_revenue,subscription_month_number
0,C101,2026-01-01,Basic,1000,NaN,NaN,NaN,1000,1
2,C101,2026-02-01,Basic,1000,1000.0,0.0,0.000000,2000,2
5,C101,2026-03-01,Pro,1800,1000.0,800.0,80.000000,3800,3
8,C101,2026-04-01,Pro,2000,1800.0,200.0,11.111111,5800,4
1,C102,2026-01-01,Pro,2500,NaN,NaN,NaN,2500,1
4,C102,2026-02-01,Pro,3000,2500.0,500.0,20.000000,5500,2
7,C102,2026-03-01,Basic,1500,3000.0,-1500.0,-50.000000,7000,3
10,C102,2026-04-01,Basic,1500,1500.0,0.0,0.000000,8500,4
3,C103,2026-01-01,Basic,800,NaN,NaN,NaN,800,1
6,C103,2026-02-01,Basic,1000,800.0,200.0,25.000000,1800,2


Which customer has generated the highest lifetime revenue by the end of April?

In [16]:
subscriptions.sort_values(by='lifetime_revenue', ascending=False).head(1)['customer_id']

10    C102
Name: customer_id, dtype: str

Which customer experienced the largest single MRR increase?

In [18]:
subscriptions.sort_values(by='mrr_change', ascending=False).head(1)['customer_id']

5    C101
Name: customer_id, dtype: str

Which customer experienced the largest single MRR decrease?

In [19]:
subscriptions.sort_values(by='mrr_change').head(1)['customer_id']

7    C102
Name: customer_id, dtype: str